# Data preparation for Llama
Stage 2 & 3

## A Overview

In [1]:
from pathlib import Path
import pandas as pd

from datasets import Dataset
from transformers import BertForSequenceClassification, BertTokenizer

import configuration

from src import hf_utils, setup
from src.models import bert

/home/ubuntu/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
bert_model_path = Path("..") / "models" / "BERT"/ "w1_o0.99888" / "1.0"
tokens_path = Path("../tokens/BERT/llama_pre_bert_set")
datasets_path = Path("..") / "data"/"splitted"

model_name = "bert-base-uncased"

device = setup.setup_device_with_seeds()

GPU: NVIDIA A100-SXM4-80GB
Memory allocated: 0.0 GB
Memory cached: 0.0 GB
Using device: cuda


In [3]:
bert_model = BertForSequenceClassification.from_pretrained(bert_model_path)
bert_tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model.to(device)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 768.02it/s]


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [29]:
df_pre_bert = pd.read_csv(datasets_path / "llama_pre_bert_sets.csv")

# For testing purposes, we can sample a smaller subset of the data
# df_pre_bert = df_pre_bert.sample(n=10000, random_state=setup.RANDOM_SEED).reset_index(drop=True)

/tmp/ipykernel_1616/1636912393.py:1: DtypeWarning: Columns (0: humanitarian_label) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pre_bert = pd.read_csv(datasets_path / "llama_pre_bert_sets.csv")


In [30]:
force_retokenize = True

if tokens_path.exists() and not force_retokenize:
    print("Loading tokenized datasets from disk...")
    pre_bert_tokenized = Dataset.load_from_disk(tokens_path)
else:
    
    ds_pre_bert = Dataset.from_pandas(df_pre_bert)
    
    # Check max token length for the 'tweet_text' column
    hf_utils.max_length_dist(df_pre_bert, 'tweet_text', bert_tokenizer)
    
    pre_bert_tokenized = hf_utils.tokenize(ds_pre_bert, bert_tokenizer, tokens_path, bert.format_dataset)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (926 > 512). Running this sequence through the model will result in indexing errors


90th percentile: 42.0
95th percentile: 46.0
99th percentile: 56.0
Absolute Maximum length: 11878


Saving the dataset (4/4 shards): 100%|██████████| 2933542/2933542 [00:03<00:00, 950454.08 examples/s] 


In [31]:
predictions, confidences = bert.predict(bert_model, pre_bert_tokenized, device, confidence_scores=True)
bert.report_metrics(pre_bert_tokenized, predictions)

Predicting:: 100%|██████████| 183347/183347 [55:38<00:00, 54.92it/s] 



Classification report:
              precision    recall  f1-score   support

       False     0.9988    0.9755    0.9870   2874375
        True     0.4425    0.9444    0.6026     59167

    accuracy                         0.9749   2933542
   macro avg     0.7206    0.9600    0.7948   2933542
weighted avg     0.9876    0.9749    0.9793   2933542



In [32]:
df_pre_bert["predicted"] = predictions
df_pre_bert["confidence"] = confidences
df_pre_bert.head()

,uid,tweet_text,informative,humanitarian_label,subset,predicted,confidence
0,37890,"In Uzbekistan, the drought wave has mostly aff...",True,rescue_volunteering_or_donation_effort,disaster,1,0.938396
1,23434,Battery Tunnel still closed - Holland Tunnel o...,True,infrastructure_and_utility_damage,disaster,1,0.957473
2,210162,.@DirectRelief is a 4⭐️ Charity Navigator orga...,True,rescue_volunteering_or_donation_effort,disaster,1,0.893836
3,178535,"If I have a specialization, what can i do to f...",True,not_humanitarian,disaster,0,0.921999
4,168358,Talking to my cousin right now from Nepal he's...,False,not_related_or_irrelevant,disaster,1,0.570526


In [ ]:
df_pre_bert.to_csv(datasets_path / "llama_pre_bert_sets_with_predictions.csv", index=False)